### Transformers


At each word, or piece of a word, in a sentence is associated a high dimensional vetor called **embedding**. The **embedding** is a representation of a token in a high dimensional space and directions in it can correspond to a semantic meaning.<br>
The aim of a transformer is to progressively adjust this embeddings to give context to the meaning of each token. Indeed, one can thing of a word which can assume different meanings depending on the context of the phrase. So the aim of the transformer is to change the embedding vector in a specific direction as a function of the context. <br>
As an example, one can considers the word *tower* and its specific embedding. This embedding points to a specfici direction in the high dimensional space. However, if in the sentence there is not only the word *tower* but also the word *Eiffel*, then it should be appropriate to *move* the embedding to an appropriate sub-space which is closer to other word embeddings, such as *France* or *Paris*. This function is performed by the **attention block** is to move information from one embedding to another.  

* Embedding usally involves the embedding vector of the word + its position in a sentence

<img src="Data/attention.gif" width=800>

In [1]:
with open("Data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [2]:
print(len(text))
print(text[:50])

1115394
First Citizen:
Before we proceed any further, hear


In [3]:
# get unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Tokenize characters

Other examples are *sentencepiece* (https://github.com/google/sentencepiece) or *tiktoken* (https://github.com/openai/tiktoken) which are subwords encoding

In [4]:
# mapping from chars to integers and vice-versa
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

# encoder: takes a string, output a list of integers
encode = lambda s: [stoi[c] for c in s]
# decoder: takes a list of integers, output a string
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("Ciao"))
print(decode(encode("Ciao")))

[15, 47, 39, 53]
Ciao


In [5]:
import torch
# tokenize input text
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:10])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [6]:
# Split input text into train and validation

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [9]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel
block_size = 8 # what is the maximum context length for predictions

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,)) # random offsets in the train or data set
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t] 
        print(f"when input is {context.tolist()} the target is: {target}") 

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is: 43
when input is [24, 43] the target is: 58
when input is [24, 43, 58] the target is: 5
when input is [24, 43, 58, 5] the target is: 57
when input is [24, 43, 58, 5, 57] the target is: 1
when input is [24, 43, 58, 5, 57, 1] the target is: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is: 39
when input is [44] the target is: 53
when input is [44, 53] the target is: 56
when input is [44, 53, 56] the target is: 1
when input is [44, 53, 56, 1] the target is: 58
when input is [44, 53, 56, 1, 58]

### Biagram model

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) -- the channel size is equal to vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B,C)
            # apply softmax to get the probabilities 
            probs = F.softmax(logits, dim=-1) # (B,C)
            # sample from the distribution 
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# generate random stuff from non trained model
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [23]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.450434446334839


In [25]:
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))


Se I m tis be h bed BRe hucul bew Jal s helas!

I's, golld.
Thondreed ait?
Pr t:
Tod.

IIViray:
ADuto beebof brs ICE sar t-by h k f mbus.
Yo wha moan
HENThitigo thes;
Awnay wenthunt! oofay nt thet o--
Mestheaitloraimf name nseat hilom moriss thethe t o selobe or sh oer conture wf imbunosoudedsuiss e


### The mathematical trick in self-attention

In [26]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C) # batch x tokens (or block size) x channels (the number of channels can be considered as the dimension of the embedding)
x.shape

torch.Size([4, 8, 2])

We would like tokens in a sequence to talk to each other. In particular, we would like to tokens in the n-th position to talk only to past tokens (up to n-1). <br>
To summarize the context of past tokens, we can pass infos like the average over the block size dimension (this is quite weak but good for now). Such average would be a vector of dimension equal to the embedding size

In [27]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t,C)    the t-th token is included
        xbow[b,t] = torch.mean(xprev,0) # averaging on the block size dimension

We can produce this more efficiently using triangular matrices

In [30]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [34]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1,keepdim=True)
xbow2 = torch.matmul(wei , x) # (B,T,T) * (B,T,C) --> (B,T,C)
torch.allclose(xbow, xbow2)

True

A third version that is easier to read can be made by using softmax

In [35]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = torch.matmul(wei , x) # (B,T,T) * (B,T,C) --> (B,T,C)
torch.allclose(xbow, xbow3)

True

In [37]:
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


With this structure we are gathering information from the past by passing the average of previous tokens (average made at each dimension of the embbedding) -- look at wei matrix. <br>
Now we would like to pass such information by weighting information in a data-dependent fashion. We don't want to wait equally all the infos from the past but rather weight them based on their importance. <br>
This data-dependent weighting of previous infos is made through a self-attention layer <br>
How do self-attention wokrs make this possible? <br>
Every single token at each position emits 2 vectors:

1. a **query** - looking at this token, what am i looking for?
2. a **key** - what this token contains?

To make query and keys talk to each other (or find affinities between them) is by making a dot product. Such dot product is an updated version of the matrix *wei* (look up), which was making just the average. Such dot product helps to learn more about each token by making it interact with the other ones, for example by highlighting the context. <br>

To produce a key and a query, two linear layers (one for query and one for key) is used. This process indipendently every token in every batch in parallel and so indipendently with the same weights. <br>
In the end the *wei* matrix will have at each position of the previous token, a weight to highlight the importance of a previous token with respect to another token. The weights are then normalized and sum to one with softmax. <br>
Once understood the context with respect to the other previous tokens in the sentence, we have to aggregate such information to each embedding of the tokens considered. To do so, instead of aggregating directly on the embedding, we use another linear layer to obtain a value **value**. 

In [41]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # bach, block_size, embed_size
x = torch.randn(B,T,C)

# single Head performing self-attention
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)

# with this form now wei is data dependent
wei = torch.matmul(q, k.transpose(-2,-1)) # (B,T,16) * (B,16,T) --> (B,T,T)

tril = torch.tril(torch.ones(T,T))

wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)


v = value(x)
out = torch.matmul(wei , v) # (B,T,T) * (B,T,16) --> (B,T,16)


Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling. The *encoder attention block* is applied in other context like sentiment analysis or translation to another language, in which all the tokens are let to talk to each other.
- "self-attention" just means that the keys and values are produced from the same source as queries (in the example above, they all come from the matrix `x`). In "cross-attention", the queries still get produced from `x`, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [42]:
# the normalization is used to keep the variance equal to one
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # scaling applied

In [43]:
k.var()

tensor(1.0449)

In [44]:
q.var()

tensor(1.0700)

In [45]:
wei.var()

tensor(1.0918)

The normalization is done to avoid to get values too high before the softmax, since the softmax tends to stress more such values and converge to one-vectors. <br>
This effect is shown below with the same vector. In the second case, we sharpen more such vector by multiplying it by 8. It can be seen that softmax sharpen even more the differences. <br>
Such normalization helps a lot especially in the inizialization of the layers of query and keys, which can induce too peaky results.

In [46]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [47]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

Usually attention is applied by applying multiple heads in parallel --> **Multi-head attention** <br>
As we have seen previously, the output of a head is (B,T,head_size). Previously was head_size=16. <br>
Usually, if we have an embed size of a certain dimension (let's say 32) and we want to use parallely 4 heads, then the head size is chosen corrispondingly to 32//4 = 8. <br>
After the batch is passed parallely in each head, the outpu is obtained by concatenating each output of the heads, so that the output as the same dimension of the input

=========


After self-attention, usually is implemented a linear layer. This is applied after the multi-head attention and individually to each token which is output to self attention. <br>
This is done to *make think individually each token* after the comunnication stage, in which tokens were passing to each other their infos. 

In [48]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [49]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(2.3842e-09), tensor(1.0000))

### Full GPT

In [53]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'mps' if torch.mps.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('Data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5091, val loss 2.5059
step 300: train loss 2.4196, val loss 2.4336
step 400: train loss 2.3500, val loss 2.3564
step 500: train loss 2.2964, val loss 2.3128
step 600: train loss 2.2406, val loss 2.2499
step 700: train loss 2.2057, val loss 2.2192
step 800: train loss 2.1633, val loss 2.1864
step 900: train loss 2.1242, val loss 2.1503
step 1000: train loss 2.1027, val loss 2.1296
step 1100: train loss 2.0700, val loss 2.1186
step 1200: train loss 2.0392, val loss 2.0804
step 1300: train loss 2.0262, val loss 2.0656
step 1400: train loss 1.9934, val loss 2.0376
step 1500: train loss 1.9706, val loss 2.0303
step 1600: train loss 1.9641, val loss 2.0463
step 1700: train loss 1.9418, val loss 2.0120
step 1800: train loss 1.9100, val loss 1.9962
step 1900: train loss 1.9078, val loss 1.9861
step 2000: train loss 1.8841, val loss 1.9936
step 2100: train loss 1.

In [54]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


Foast.

MERCUTEN ELLIZABETH:
Uf Wilwick.

HENRY BOLINGBROKE:
My reverecomme the nebesser, reads Lord,
Who is make that at I coubt, everingling
That dliht thy winger to see awise the letsts love's slemm me:
Than what you suffeer toogny!
That proply enstriaght with a seasn.
Why, they four tleadss,--

VOMINIUS:

Letsemer:
You what, in mamay love you. O, evounjust.
I'll that sholl recimandeds wontime;
That pad to this me. Mine, that read;
Werses the been your done? abott, as town this dray.

ROMEO:
O, upon to death! him not this bornorow-prince.
Myself would with the curtent, Dercome?
As in you eat the waster!
As the day, my solders worsuo most fathrove,
My settorne but guary wisith his battal: and broth of heaving,
Coer whose rest pein.

HENMONRA:
Gi, my verself:
Her light me; our incae, whou command
Os must mair'st temoust me
In revoy I knows graces with her.

HORTENS:
Stay,--hing,
I'll facestion peop of thou do
As hath lay. Woust is daughter awaster
Hast was and holy toush me, gurself 